# Train the waste detector (Colab / Kaggle GPU)

1. Runtime → Change runtime type → **GPU**.
2. On the laptop run `bash models/vision/make_bundle.sh` and upload `models/vision/out/waste_segregation.zip` with the file panel on the left. The repo is private, so it is not cloned here.
3. Set `LICENCES` in the data cell (see below), then run the cells in order. At the end, download `best.pt` and `per_class.json`, and attach `best.pt` to a GitHub Release.

Colab's own CUDA PyTorch is used; only Ultralytics and small tools are added with uv.

**Licences of the TACO images are mixed** (see THIRD_PARTY.md and `data/taco_licences.md`): `all` keeps every image, `known` drops images with no licence field, `odbl` keeps only OpenLitterMap ODbL images. Choose before training weights you will publish.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!unzip -q -o waste_segregation.zip -d waste_segregation
%cd waste_segregation

In [ ]:
!pip -q install uv
!uv pip install --system "ultralytics==8.4.202" "pyyaml>=6.0" "pillow>=11"
import os

os.environ["YOLO_AUTOINSTALL"] = "false"
from ultralytics import settings

settings.update({"sync": False})

In [ ]:
# Data: TACO is downloaded here. Add other COCO datasets with --coco NAME=path/annotations.json
LICENCES = "all"  # "all", "known" or "odbl": see the note at the top
!python data/scripts/download_taco.py
!python data/scripts/prepare.py --licences {LICENCES}
!cat data/processed/waste/report.json

In [ ]:
!python models/vision/train.py --data data/processed/waste/dataset.yaml --epochs 100 --batch 32

In [ ]:
from google.colab import files  # on Kaggle, use the Output tab instead

files.download("models/vision/runs/yolo26n_waste/weights/best.pt")
files.download("models/vision/runs/yolo26n_waste/per_class.json")